# Experiments - SKILL 2025

In [1]:
import os
import pandas as pd
from config import algorithm_strategy_pairs
from utils.simulation_utils import get_directory_for_algorithm

In [2]:
def get_sorted_results_by_regret(arms, timestep=1000000):
    """
    Function to calculate and return sorted results for given arms and timestep by "Average Regret".

    Parameters:
        arms (list): List of arm probabilities.
        timestep (int): The timestep to filter results. Default is 1000000.

    Returns:
        pandas.DataFrame: A sorted DataFrame of results by "Average Regret" in descending order.
    """
    # Initialize results dictionary
    results = {}

    # Iterate through each algorithm and strategy pair
    for algorithm, strategy in algorithm_strategy_pairs:
        combination_name = "_".join(str(int(arm * 1000)) for arm in arms)
        results_dir = get_directory_for_algorithm(algorithm, strategy["params"]).replace('/src', '', 1)
        detailed_results_path = os.path.join(results_dir, f'average_results_{combination_name}.csv')

        if os.path.exists(detailed_results_path):
            df = pd.read_csv(detailed_results_path)
            filtered_data = df[df['Timestep'] == timestep]
            strategy_suffix = "_".join(f"{k}-{v}" for k, v in strategy["params"].items()) or "default"
            results_key = f"{algorithm.name}_{strategy_suffix}"
            if not filtered_data.empty:
                results[results_key] = filtered_data.iloc[0]  # Take the first row of filtered data

    # Convert results dictionary to a DataFrame
    results_df = pd.DataFrame.from_dict(results, orient='index')

    # Sort the DataFrame by "Average Regret" in descending order
    sorted_results = results_df.sort_values(by="Average Total Reward", ascending=False)

    return sorted_results


In [3]:
# Define a function to calculate the ratio of choosing a suboptimal arm
def calculate_suboptimal_ratio(arms, timestep):
    results = {}
    for algorithm, strategy in algorithm_strategy_pairs:
        combination_name = "_".join(str(int(arm * 1000)) for arm in arms)
        results_dir = get_directory_for_algorithm(algorithm, strategy["params"]).replace('/src', '', 1)
        detailed_results_path = os.path.join(results_dir, f'average_results_{combination_name}.csv')

        if os.path.exists(detailed_results_path):
            df = pd.read_csv(detailed_results_path)
            filtered_data = df[df['Timestep'] == timestep]
            if not filtered_data.empty:
                suboptimal_ratio = filtered_data['Average Suboptimal Arms'] / timestep
                strategy_suffix = "_".join(f"{k}-{v}" for k, v in strategy["params"].items()) or "default"
                results_key = f"{algorithm.name}_{strategy_suffix}"
                results[results_key] = suboptimal_ratio.values[0]
    return results

In [4]:
def calculate_var_for_arms(arms, timestep=1000000):
    """
    Calculate results for given arms and timestep.

    Parameters:
        arms (list): List of arm probabilities.
        timestep (int): The timestep to filter results. Default is 1000000.

    Returns:
        pandas.DataFrame: A DataFrame containing the results.
    """
    # Initialize results dictionary
    results = {}

    # Iterate through each algorithm and strategy pair
    for algorithm, strategy in algorithm_strategy_pairs:
        combination_name = "_".join(str(int(arm * 1000)) for arm in arms)
        results_dir = get_directory_for_algorithm(algorithm, strategy["params"]).replace('/src', '', 1)
        detailed_results_path = os.path.join(results_dir, f'results_{combination_name}.csv')

        if os.path.exists(detailed_results_path):
            df = pd.read_csv(detailed_results_path)
            filtered_data = df[df['Timestep'] == timestep]

            if not filtered_data.empty:
                reward_variance = filtered_data['Total Reward'].var()
                regret_variance = filtered_data['Total Regret'].var()
                avg_reward = filtered_data['Total Reward'].mean()
                avg_regret = filtered_data['Total Regret'].mean()
            else:
                print(f"No data for timestep {timestep} in {detailed_results_path}")  # Debug print
                reward_variance = None
                regret_variance = None
                avg_reward = None
                avg_regret = None

            strategy_suffix = "_".join(f"{k}-{v}" for k, v in strategy["params"].items()) or "default"
            results_key = f"{algorithm.name}_{strategy_suffix}"

            # Add average reward, regret, and variances
            results[results_key] = {
                "Average Reward": avg_reward,
                "Average Regret": avg_regret,
                "Reward Variance": reward_variance,
                "Regret Variance": regret_variance
            }
        else:
            print(f"File not found: {detailed_results_path}")  # Debug print
            # Add placeholder for missing data
            strategy_suffix = "_".join(f"{k}-{v}" for k, v in strategy["params"].items()) or "default"
            results_key = f"{algorithm.name}__{strategy_suffix}"
            results[results_key] = {
                "Average Reward": None,
                "Average Regret": None,
                "Reward Variance": None,
                "Regret Variance": None
            }

    # Convert results dictionary to a DataFrame
    results_df = pd.DataFrame.from_dict(results, orient='index').reset_index()
    results_df.columns = ['Algorithm', 'Average Reward', 'Average Regret', 'Reward Variance', 'Regret Variance']

    print("Final results DataFrame:")  # Debug print
    print(results_df)  # Debug print

    return results_df


## Scenario A: Baseline

In [5]:
sorted_results_by_regret = get_sorted_results_by_regret([0.8, 0.9], timestep=1000000)
print(sorted_results_by_regret)

var_for_arms= calculate_var_for_arms([0.8, 0.9], timestep=1000000)
print(var_for_arms)

# Calculate the ratio for Timestep=10000
suboptimal_ratio_10000 = calculate_suboptimal_ratio([0.8, 0.9], 10000)
print("Suboptimal Ratio for Timestep=10000:")
print(suboptimal_ratio_10000)

# Calculate the ratio for Timestep=1000000
suboptimal_ratio_1000000 = calculate_suboptimal_ratio([0.8, 0.9], 1000000)
print("Suboptimal Ratio for Timestep=1000000:")
print(suboptimal_ratio_1000000)

                                  Timestep  Average Total Reward  \
ETC_exploration_rounds-100       1000000.0             899997.98   
UCB-Tuned_default                1000000.0             899977.37   
EUCBV_rho-0.5                    1000000.0             899961.27   
PAC-UCB_c-1_b-1_q-1.3_beta-0.05  1000000.0             899907.85   
ETC_exploration_rounds-1000      1000000.0             899907.29   
UCB-V_theta-1_c-1_b-1            1000000.0             899898.45   
UCB_default                      1000000.0             899766.19   
Greedy_epsilon-0.005             1000000.0             899581.43   
Greedy_epsilon-0.01              1000000.0             899422.38   
ETC_exploration_rounds-10000     1000000.0             899003.34   
Greedy_epsilon-0.05              1000000.0             897483.41   
Greedy_epsilon-0.1               1000000.0             895001.31   
ETC_exploration_rounds-10        1000000.0             893010.47   
ETC_exploration_rounds-100000    1000000.0      

## Scenario B: Low-Variance Micro-Gap

In [6]:
sorted_results_by_regret = get_sorted_results_by_regret([0.895, 0.9], timestep=1000000)
print(sorted_results_by_regret)

var_for_arms= calculate_var_for_arms([0.895, 0.9], timestep=1000000)
print(var_for_arms)

# Calculate the ratio for Timestep=10000
suboptimal_ratio_10000 = calculate_suboptimal_ratio([0.895, 0.9], 10000)
print("Suboptimal Ratio for Timestep=10000:")
print(suboptimal_ratio_10000)

# Calculate the ratio for Timestep=1000000
suboptimal_ratio_1000000 = calculate_suboptimal_ratio([0.895, 0.9], 1000000)
print("Suboptimal Ratio for Timestep=1000000:")
print(suboptimal_ratio_1000000)

                                  Timestep  Average Total Reward  \
ETC_exploration_rounds-10000     1000000.0             899856.46   
UCB-Tuned_default                1000000.0             899795.54   
Greedy_epsilon-0.1               1000000.0             899664.48   
UCB-V_theta-1_c-1_b-1            1000000.0             899661.09   
Greedy_epsilon-0.05              1000000.0             899647.49   
PAC-UCB_c-1_b-1_q-1.3_beta-0.05  1000000.0             899615.84   
EUCBV_rho-0.5                    1000000.0             899546.35   
ETC_exploration_rounds-100000    1000000.0             899508.90   
Greedy_epsilon-0.01              1000000.0             898907.05   
UCB_default                      1000000.0             898880.02   
Greedy_epsilon-0.5               1000000.0             898737.76   
ETC_exploration_rounds-1000      1000000.0             898712.26   
Greedy_epsilon-0.005             1000000.0             898574.61   
ETC_exploration_rounds-100       1000000.0      

## Scenario C: High-Variance Micro-Gap

In [7]:
sorted_results_by_regret = get_sorted_results_by_regret([0.89, 0.895], timestep=1000000)
print(sorted_results_by_regret)

var_for_arms= calculate_var_for_arms([0.89, 0.895], timestep=1000000)
print(var_for_arms)

# Calculate the ratio for Timestep=10000
suboptimal_ratio_10000 = calculate_suboptimal_ratio([0.89, 0.895], 10000)
print("Suboptimal Ratio for Timestep=10000:")
print(suboptimal_ratio_10000)

# Calculate the ratio for Timestep=1000000
suboptimal_ratio_1000000 = calculate_suboptimal_ratio([0.89, 0.895], 1000000)
print("Suboptimal Ratio for Timestep=1000000:")
print(suboptimal_ratio_1000000)

                                  Timestep  Average Total Reward  \
ETC_exploration_rounds-10000     1000000.0             894820.57   
UCB-Tuned_default                1000000.0             894766.72   
Greedy_epsilon-0.05              1000000.0             894691.56   
Greedy_epsilon-0.1               1000000.0             894625.65   
UCB-V_theta-1_c-1_b-1            1000000.0             894624.46   
PAC-UCB_c-1_b-1_q-1.3_beta-0.05  1000000.0             894574.50   
EUCBV_rho-0.5                    1000000.0             894514.12   
ETC_exploration_rounds-100000    1000000.0             894493.54   
ETC_exploration_rounds-1000      1000000.0             893831.50   
UCB_default                      1000000.0             893819.56   
Greedy_epsilon-0.01              1000000.0             893787.47   
Greedy_epsilon-0.5               1000000.0             893722.15   
Greedy_epsilon-0.005             1000000.0             893225.36   
ETC_exploration_rounds-100       1000000.0      